# DeepDoc + VietOCR: Kaggle T4 x2 hybrid/batched OCR

This notebook reads keyframes recursively from a Kaggle input dataset, runs ONNX text detection plus batched VietOCR recognition on two T4 GPUs, and writes one `ocr_results.json` file. Enable **Internet** and select **GPU T4 x2** before running every cell from top to bottom.


In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError(
        f"Expected Kaggle GPU T4 x2, but found {torch.cuda.device_count()} CUDA GPU(s). "
        "Open Notebook options and select Accelerator: GPU T4 x2."
    )

for device_id in range(2):
    print(f"GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
print(f"PyTorch CUDA: {torch.version.cuda}")


GPU 0: Tesla T4
GPU 1: Tesla T4
PyTorch CUDA: 12.8


In [2]:
from pathlib import Path
import re


def filter_upstream_requirements(lines: list[str]) -> list[str]:
    excluded = {"torch", "onnxruntime", "onnxruntime-gpu"}
    selected = []
    for raw_line in lines:
        requirement = raw_line.strip()
        if not requirement or requirement.startswith("#"):
            continue
        package = re.split(r"[<>=!~;\s\[]", requirement, maxsplit=1)[0]
        package = package.lower().replace("_", "-")
        if package not in excluded:
            selected.append(requirement)
    return selected


def onnxruntime_gpu_requirement(cuda_version: str | None) -> str:
    cuda_major = cuda_version.split(".", 1)[0] if cuda_version else ""
    versions = {"12": "1.26.0", "13": "1.28.0"}
    if cuda_major not in versions:
        raise RuntimeError(
            f"This notebook supports Kaggle CUDA 12 or 13; found {cuda_version!r}."
        )
    return f"onnxruntime-gpu=={versions[cuda_major]}"


def build_pinned_clone_commands(
    repo_url: str, repo_dir: str | Path, commit: str
) -> list[list[str]]:
    repo_dir = str(repo_dir)
    return [
        ["git", "init", repo_dir],
        ["git", "-C", repo_dir, "remote", "add", "origin", repo_url],
        ["git", "-C", repo_dir, "fetch", "--depth", "1", "origin", commit],
        ["git", "-C", repo_dir, "checkout", "--detach", "FETCH_HEAD"],
        ["git", "-C", repo_dir, "lfs", "pull"],
    ]


In [3]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/hoaivannguyen/deepdoc_vietocr.git"
REPO_COMMIT = "8f7906fd4e78cd0324532298e9062b031b4a320c"
REPO_DIR = Path("/kaggle/working/deepdoc_vietocr")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
for command in build_pinned_clone_commands(REPO_URL, REPO_DIR, REPO_COMMIT):
    subprocess.run(command, check=True)

requirements = filter_upstream_requirements(
    (REPO_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
)
ort_requirement = onnxruntime_gpu_requirement(torch.version.cuda)
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *requirements,
        ort_requirement,
        "strenum",
        "trio==0.34.0",
        "vietocr==0.3.13",
    ],
    check=True,
)

import onnxruntime as ort

providers = ort.get_available_providers()
if "CUDAExecutionProvider" not in providers:
    raise RuntimeError(f"ONNX Runtime cannot use CUDA. Available providers: {providers}")
print(f"ONNX Runtime providers: {providers}")


Initialized empty Git repository in /kaggle/working/deepdoc_vietocr/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
From https://github.com/hoaivannguyen/deepdoc_vietocr
 * branch            8f7906fd4e78cd0324532298e9062b031b4a320c -> FETCH_HEAD
HEAD is now at 8f7906f readme test update


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
from pathlib import Path


def _replace_required(path: Path, original: str, replacement: str) -> None:
    source = path.read_text(encoding="utf-8")
    if replacement in source:
        return
    if original not in source:
        raise RuntimeError(f"Cannot patch upstream source; expected marker missing from {path}")
    path.write_text(source.replace(original, replacement), encoding="utf-8")


def patch_repo_for_kaggle_gpu(repo_dir: str | Path) -> None:
    recognizer = Path(repo_dir) / "module" / "ocr.py"
    _replace_required(
        recognizer,
        'config[\'weights\'] = r"vietocr\\weight\\vgg_seq2seq.pth"',
        'config[\'weights\'] = "vietocr/weight/vgg_seq2seq.pth"',
    )
    _replace_required(
        recognizer,
        "config['device'] = 'cpu'",
        "config['device'] = f'cuda:{0 if device_id is None else device_id}'",
    )
    _replace_required(
        recognizer,
        "        self.detector = Predictor(config)",
        "        self.batch_size = max(1, int(os.environ.get('VIETOCR_BATCH_SIZE', '32')))\n"
        "        self.detector = Predictor(config)",
    )
    _replace_required(
        recognizer,
        (
            "    def __call__(self, img_list):\n"
            "        results = []\n"
            "        for img in img_list:\n"
            "            # Ensure PIL Image\n"
            "            if isinstance(img, np.ndarray):\n"
            "                img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))\n"
            "            text = self.detector.predict(img)\n"
            "            results.append((text, 1.0))\n"
            "        return results, 0.0"
        ),
        (
            "    def __call__(self, img_list):\n"
            "        start_time = time.time()\n"
            "        prepared = []\n"
            "        for img in img_list:\n"
            "            if isinstance(img, np.ndarray):\n"
            "                img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))\n"
            "            prepared.append(img)\n"
            "        texts = []\n"
            "        for start in range(0, len(prepared), self.batch_size):\n"
            "            batch = prepared[start:start + self.batch_size]\n"
            "            texts.extend(self.detector.predict_batch(batch))\n"
            "        return [(text, 1.0) for text in texts], time.time() - start_time"
        ),
    )
    _replace_required(
        recognizer,
        "    loaded_model = (sess, run_options)",
        (
            '    if cuda_is_available() and sess.get_providers()[0] != "CUDAExecutionProvider":\n'
            '        raise RuntimeError(\n'
            '            f"ONNX text detection fell back from CUDA: {sess.get_providers()}"\n'
            '        )\n'
            "    loaded_model = (sess, run_options)"
        ),
    )


In [5]:
RUNNER_SOURCE = r'''
from __future__ import annotations

import argparse
from concurrent.futures import ThreadPoolExecutor
import json
import os
from pathlib import Path
from queue import Empty, Full, Queue
import re
import threading
import time

import numpy as np
from PIL import Image


os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")

SUPPORTED_IMAGE_SUFFIXES = {
    ".bmp",
    ".jpeg",
    ".jpg",
    ".png",
    ".tif",
    ".tiff",
    ".webp",
}
KEYFRAME_PATTERN = re.compile(
    r"^(?P<video_id>.+)_s(?P<shot_id>\d+)_f(?P<frame_id>\d+)$"
)


def parse_keyframe_name(filename: str | Path) -> dict[str, str]:
    stem = Path(filename).stem
    match = KEYFRAME_PATTERN.fullmatch(stem)
    if match is None:
        raise ValueError(
            f"Invalid keyframe filename {Path(filename).name!r}; expected "
            "<video_id>_s<shot_id>_f<frame_id>.<image extension>"
        )
    return {
        "video_id": match.group("video_id"),
        "shot_id": str(int(match.group("shot_id"))),
        "frame_id": str(int(match.group("frame_id"))),
    }


def record_key(record: dict[str, str]) -> tuple[str, str, str]:
    return record["video_id"], record["shot_id"], record["frame_id"]


def normalize_ocr_text(fragments: list[str | None]) -> str:
    normalized = [
        " ".join(str(fragment).split())
        for fragment in fragments
        if fragment is not None
    ]
    return " ".join(fragment for fragment in normalized if fragment)


def iter_keyframe_events(
    inputs: str | Path,
    start_image: int = 1,
    end_image: int | None = None,
):
    inputs = Path(inputs)
    if not inputs.exists():
        raise FileNotFoundError(f"Input path does not exist: {inputs}")
    if start_image < 1:
        raise ValueError("start_image must be at least 1")
    if end_image is not None and end_image < start_image:
        raise ValueError("end_image must be greater than or equal to start_image")
    if inputs.is_file():
        candidates = [inputs]
    else:
        candidates = sorted(
            (
            path
            for path in inputs.rglob("*")
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
            ),
            key=lambda path: path.as_posix(),
        )
    if not candidates:
        raise RuntimeError(f"No supported keyframe images found under {inputs}")

    candidates = candidates[start_image - 1 : end_image]
    if not candidates:
        range_end = "end" if end_image is None else str(end_image)
        raise RuntimeError(
            f"Image range {start_image}-{range_end} does not overlap "
            f"the available images under {inputs}"
        )

    seen = set()
    for path in candidates:
        try:
            key = record_key(parse_keyframe_name(path.name))
        except ValueError as error:
            yield None, {"path": str(path), "error": str(error)}
            continue
        if key in seen:
            yield None, (
                {
                    "path": str(path),
                    "error": f"Duplicate video/shot/frame ID: {'/'.join(key)}",
                }
            )
            continue
        seen.add(key)
        yield path, None


def load_processed_keys(checkpoint_path: str | Path) -> set[tuple[str, str, str]]:
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return set()
    processed = set()
    valid_end = 0
    repair_trailing_line = False
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint:
        line_number = 0
        while True:
            line = checkpoint.readline()
            if not line:
                break
            line_number += 1
            if not line.strip():
                valid_end = checkpoint.tell()
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                if not line.endswith("\n") and checkpoint.read(1) == "":
                    repair_trailing_line = True
                    break
                raise RuntimeError(
                    f"Invalid checkpoint JSON on line {line_number}: {checkpoint_path}"
                ) from error
            processed.add(record_key(record))
            valid_end = checkpoint.tell()
    if repair_trailing_line:
        with checkpoint_path.open("r+", encoding="utf-8") as checkpoint:
            checkpoint.seek(valid_end)
            checkpoint.truncate()
    return processed


def process_frame_group(
    ocr,
    paths: list[Path],
    device_id: int,
    timings: dict | None = None,
):
    group_started = time.perf_counter()
    detection_seconds = 0.0
    recognition_seconds = 0.0
    frames = []
    all_crops = []
    failures = []

    def store_timings() -> None:
        if timings is not None:
            timings.update(
                {
                    "frames": len(paths),
                    "crops": len(all_crops),
                    "detect_seconds": detection_seconds,
                    "recognize_seconds": recognition_seconds,
                    "total_seconds": time.perf_counter() - group_started,
                }
            )

    for path in paths:
        try:
            with Image.open(path) as image:
                image_array = np.array(image.convert("RGB"))
            detection_started = time.perf_counter()
            try:
                boxes, _ = ocr.text_detector[device_id](image_array)
            finally:
                detection_seconds += time.perf_counter() - detection_started
            sorted_boxes = [] if boxes is None else ocr.sorted_boxes(boxes)
            crop_start = len(all_crops)
            for box in sorted_boxes:
                crop = ocr.get_rotate_crop_image(
                    image_array,
                    np.array(box, copy=True),
                )
                all_crops.append(crop)
            frames.append(
                {
                    "path": path,
                    "ids": parse_keyframe_name(path.name),
                    "crop_start": crop_start,
                    "crop_end": len(all_crops),
                }
            )
        except Exception as error:
            failures.append({"path": str(path), "error": repr(error)})

    if all_crops:
        recognition_failed = False
        recognition_started = time.perf_counter()
        try:
            recognized = ocr.recognize_batch(all_crops, device_id=device_id)
            if len(recognized) != len(all_crops):
                raise RuntimeError(
                    f"VietOCR returned {len(recognized)} result(s) for "
                    f"{len(all_crops)} crop(s)"
                )
        except Exception as error:
            failures.extend(
                {"path": str(frame["path"]), "error": repr(error)}
                for frame in frames
            )
            recognition_failed = True
        finally:
            recognition_seconds += time.perf_counter() - recognition_started
        if recognition_failed:
            store_timings()
            return [], failures
    else:
        recognized = []

    records = []
    for frame in frames:
        fragments = recognized[frame["crop_start"] : frame["crop_end"]]
        records.append(
            {
                **frame["ids"],
                "ocr_text": normalize_ocr_text(fragments),
            }
        )
    store_timings()
    return records, failures


def run_ocr_pipeline(
    ocr,
    keyframe_events,
    checkpoint_path: str | Path,
    failures_path: str | Path,
    gpu_count: int = 2,
    frame_batch_size: int = 8,
    progress_every: int = 100,
) -> dict[str, int]:
    if gpu_count != 2:
        raise ValueError("This Kaggle runner requires exactly 2 GPUs")
    if frame_batch_size < 1:
        raise ValueError("frame_batch_size must be at least 1")

    checkpoint_path = Path(checkpoint_path)
    failures_path = Path(failures_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    failures_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint_path.touch(exist_ok=True)
    failures_path.touch(exist_ok=True)

    processed_keys = load_processed_keys(checkpoint_path)
    counters = {
        "processed": 0,
        "skipped": 0,
        "failed": 0,
    }
    write_lock = threading.Lock()
    device_queues = [
        Queue(maxsize=max(2, frame_batch_size * 2)) for _ in range(gpu_count)
    ]
    device_batch_counts = [0 for _ in range(gpu_count)]
    sentinel = object()
    cancel_event = threading.Event()

    with checkpoint_path.open("a", encoding="utf-8") as checkpoint, failures_path.open(
        "a", encoding="utf-8"
    ) as failure_log:

        def report_progress(previous_total: int) -> None:
            current_total = sum(counters.values())
            if progress_every and current_total // progress_every > previous_total // progress_every:
                print(
                    f"Handled {current_total} keyframes "
                    f"({counters['skipped']} resumed, {counters['failed']} failed)"
                )

        def append_failure(failure: dict[str, str]) -> None:
            previous_total = sum(counters.values())
            failure_log.write(
                json.dumps(failure, ensure_ascii=False, separators=(",", ":")) + "\n"
            )
            failure_log.flush()
            counters["failed"] += 1
            report_progress(previous_total)

        def commit_group(device_id: int, group: list[Path]) -> None:
            timings = {}
            records, failures = process_frame_group(
                ocr,
                group,
                device_id,
                timings=timings,
            )
            with write_lock:
                previous_total = sum(counters.values())
                for record in records:
                    checkpoint.write(
                        json.dumps(record, ensure_ascii=False, separators=(",", ":"))
                        + "\n"
                    )
                checkpoint.flush()
                for failure in failures:
                    failure_log.write(
                        json.dumps(failure, ensure_ascii=False, separators=(",", ":"))
                        + "\n"
                    )
                failure_log.flush()
                counters["processed"] += len(records)
                counters["failed"] += len(failures)
                report_progress(previous_total)
                device_batch_counts[device_id] += 1
                print(
                    f"GPU {device_id} | batch {device_batch_counts[device_id]} | "
                    f"frames {timings['frames']} | crops {timings['crops']} | "
                    f"detect {timings['detect_seconds']:.2f}s | "
                    f"recognize {timings['recognize_seconds']:.2f}s | "
                    f"total {timings['total_seconds']:.2f}s",
                    flush=True,
                )

        def worker(device_id: int) -> None:
            group = []
            device_queue = device_queues[device_id]
            try:
                while True:
                    if cancel_event.is_set():
                        return
                    item = device_queue.get()
                    try:
                        if item is sentinel:
                            if group and not cancel_event.is_set():
                                commit_group(device_id, group)
                            return
                        group.append(item)
                        if len(group) == frame_batch_size:
                            commit_group(device_id, group)
                            group = []
                    finally:
                        device_queue.task_done()
            except BaseException:
                cancel_event.set()
                raise

        with ThreadPoolExecutor(max_workers=gpu_count) as executor:
            futures = [
                executor.submit(worker, device_id)
                for device_id in range(gpu_count)
            ]

            def raise_completed_worker_error() -> None:
                for future in futures:
                    if future.done():
                        error = future.exception()
                        if error is not None:
                            raise error

            def enqueue_for_worker(device_id: int, item) -> None:
                device_queue = device_queues[device_id]
                while True:
                    raise_completed_worker_error()
                    try:
                        device_queue.put(item, timeout=0.1)
                        return
                    except Full:
                        continue

            def drain_pending_items(device_queue: Queue) -> None:
                while True:
                    try:
                        device_queue.get_nowait()
                    except Empty:
                        return
                    else:
                        device_queue.task_done()

            def stop_worker(device_id: int) -> None:
                device_queue = device_queues[device_id]
                future = futures[device_id]
                while not future.done():
                    try:
                        device_queue.put(sentinel, timeout=0.1)
                        return
                    except Full:
                        if cancel_event.is_set():
                            drain_pending_items(device_queue)

            dispatch_index = 0
            producer_error = None
            try:
                for path, discovery_failure in keyframe_events:
                    if discovery_failure is not None:
                        with write_lock:
                            append_failure(discovery_failure)
                        continue
                    if path is None:
                        raise RuntimeError("Keyframe event contains neither a path nor a failure")
                    path = Path(path)
                    key = record_key(parse_keyframe_name(path.name))
                    if key in processed_keys:
                        with write_lock:
                            previous_total = sum(counters.values())
                            counters["skipped"] += 1
                            report_progress(previous_total)
                        continue
                    enqueue_for_worker(dispatch_index % gpu_count, path)
                    dispatch_index += 1
            except BaseException as error:
                producer_error = error
                cancel_event.set()
            finally:
                if cancel_event.is_set():
                    for device_queue in device_queues:
                        drain_pending_items(device_queue)
                for device_id in range(gpu_count):
                    stop_worker(device_id)

            worker_error = None
            for future in futures:
                try:
                    future.result()
                except BaseException as error:
                    if worker_error is None:
                        worker_error = error
            if producer_error is not None:
                raise producer_error
            if worker_error is not None:
                raise worker_error

    return counters


def finalize_jsonl(checkpoint_path: str | Path, output_json: str | Path) -> None:
    checkpoint_path = Path(checkpoint_path)
    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    temporary_output = output_json.with_suffix(output_json.suffix + ".tmp")

    with checkpoint_path.open("r", encoding="utf-8") as checkpoint, temporary_output.open(
        "w", encoding="utf-8"
    ) as output:
        output.write("[\n")
        first = True
        for line_number, line in enumerate(checkpoint, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    f"Invalid checkpoint JSON on line {line_number}: {checkpoint_path}"
                ) from error
            if not first:
                output.write(",\n")
            output.write("  " + json.dumps(record, ensure_ascii=False))
            first = False
        output.write("\n]\n")
    os.replace(temporary_output, output_json)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--inputs", required=True)
    parser.add_argument("--output_json", required=True)
    parser.add_argument("--checkpoint_jsonl", required=True)
    parser.add_argument("--failures_jsonl", required=True)
    parser.add_argument("--gpus", type=int, default=2)
    parser.add_argument("--frame_batch_size", type=int, default=8)
    parser.add_argument("--recognition_batch_size", type=int, default=32)
    parser.add_argument("--start_image", type=int, default=1)
    parser.add_argument("--end_image", type=int, default=None)
    parser.add_argument("--progress_every", type=int, default=100)
    args = parser.parse_args()

    if args.gpus != 2:
        raise RuntimeError("Set --gpus 2 for the Kaggle T4 x2 runtime")
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    os.environ["VIETOCR_BATCH_SIZE"] = str(args.recognition_batch_size)

    import torch

    if torch.cuda.device_count() < args.gpus:
        raise RuntimeError(
            f"Expected 2 CUDA GPUs but found {torch.cuda.device_count()}. "
            "Select Kaggle Accelerator: GPU T4 x2."
        )

    from module.ocr import OCR

    keyframe_events = iter_keyframe_events(
        args.inputs,
        start_image=args.start_image,
        end_image=args.end_image,
    )
    range_end = "end" if args.end_image is None else args.end_image
    print(f"Processing sorted images {args.start_image} through {range_end} (inclusive)")
    print(
        f"Using 2 GPUs, {args.frame_batch_size} frames per hybrid group, "
        f"and recognition batches of {args.recognition_batch_size} crops"
    )
    ocr = OCR()
    if len(ocr.text_detector) < args.gpus or len(ocr.text_recognizer) < args.gpus:
        raise RuntimeError("DeepDoc did not initialize one detector and recognizer per GPU")

    summary = run_ocr_pipeline(
        ocr=ocr,
        keyframe_events=keyframe_events,
        checkpoint_path=args.checkpoint_jsonl,
        failures_path=args.failures_jsonl,
        gpu_count=args.gpus,
        frame_batch_size=args.frame_batch_size,
        progress_every=args.progress_every,
    )
    finalize_jsonl(args.checkpoint_jsonl, args.output_json)
    print(f"OCR complete: {summary}")
    print(f"JSON output: {args.output_json}")


if __name__ == "__main__":
    main()
'''


In [6]:
patch_repo_for_kaggle_gpu(REPO_DIR)
runner_path = REPO_DIR / "t_ocr.py"
compile(RUNNER_SOURCE, str(runner_path), "exec")
runner_path.write_text(RUNNER_SOURCE, encoding="utf-8")
print("Patched DeepDoc for cuda:0/cuda:1 and installed the streaming batched t_ocr.py runner.")


Patched DeepDoc for cuda:0/cuda:1 and installed the streaming batched t_ocr.py runner.


In [7]:
from pathlib import Path
import shutil
import sys


def cleanup_ocr_artifacts(
    checkpoint_path: str | Path,
    repo_dir: str | Path,
) -> None:
    Path(checkpoint_path).unlink(missing_ok=True)
    resolved_repo = Path(repo_dir).resolve()
    if resolved_repo.name != "deepdoc_vietocr":
        raise ValueError(f"Refusing to delete unexpected repository path: {resolved_repo}")
    if resolved_repo.exists():
        shutil.rmtree(resolved_repo)


def build_ocr_command(
    inputs: str | Path,
    output_json: str | Path,
    checkpoint_jsonl: str | Path,
    failures_jsonl: str | Path,
    frame_batch_size: int = 8,
    recognition_batch_size: int = 32,
    start_image: int = 1,
    end_image: int | None = None,
) -> list[str]:
    command = [
        sys.executable,
        "t_ocr.py",
        "--inputs",
        str(inputs),
        "--output_json",
        str(output_json),
        "--checkpoint_jsonl",
        str(checkpoint_jsonl),
        "--failures_jsonl",
        str(failures_jsonl),
        "--gpus",
        "2",
        "--frame_batch_size",
        str(frame_batch_size),
        "--recognition_batch_size",
        str(recognition_batch_size),
        "--start_image",
        str(start_image),
    ]
    if end_image is not None:
        command.extend(["--end_image", str(end_image)])
    return command


## Configure Kaggle input

Change only `INPUT_DIR` to the folder containing your keyframe images. The runner searches its subfolders recursively. Output and checkpoints are written under `/kaggle/working`.


In [8]:
from pathlib import Path

# CHANGE THIS LINE to your attached Kaggle dataset folder.
INPUT_DIR = Path("/kaggle/input/datasets/trieu241007/keyframes-p1/keyframes")

OUTPUT_JSON = Path("/kaggle/working/ocr_results.json")
CHECKPOINT_JSONL = Path("/kaggle/working/ocr_results.checkpoint.jsonl")
FAILURES_JSONL = Path("/kaggle/working/ocr_failures.jsonl")
FRAME_BATCH_SIZE = 64
RECOGNITION_BATCH_SIZE = 1024
# 1-based, inclusive positions after sorting all image paths alphabetically.
START_IMAGE = 103747
END_IMAGE = 207492
if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Edit INPUT_DIR first. Kaggle input folder does not exist: {INPUT_DIR}"
    )
print(f"Input: {INPUT_DIR}")
print(f"Final JSON: {OUTPUT_JSON}")


Input: /kaggle/input/datasets/trieu241007/keyframes-p1/keyframes
Final JSON: /kaggle/working/ocr_results.json


In [9]:
import shutil
from pathlib import Path

UPLOADED_CHECKPOINT = Path(
    "/kaggle/input/datasets/trieu241007/ddddddddddddddddddddd/ocr_results.checkpoint.jsonl"
)

if not UPLOADED_CHECKPOINT.exists():
    raise FileNotFoundError(UPLOADED_CHECKPOINT)

shutil.copy2(UPLOADED_CHECKPOINT, CHECKPOINT_JSONL)
FAILURES_JSONL.unlink(missing_ok=True)

print("Restored checkpoint:", CHECKPOINT_JSONL)

Restored checkpoint: /kaggle/working/ocr_results.checkpoint.jsonl


In [10]:
import subprocess

command = build_ocr_command(
    INPUT_DIR,
    OUTPUT_JSON,
    CHECKPOINT_JSONL,
    FAILURES_JSONL,
    frame_batch_size=FRAME_BATCH_SIZE,
    recognition_batch_size=RECOGNITION_BATCH_SIZE,
    start_image=START_IMAGE,
    end_image=END_IMAGE,
)
environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"
environment["VIETOCR_BATCH_SIZE"] = str(RECOGNITION_BATCH_SIZE)
completed = subprocess.run(command, cwd=REPO_DIR, env=environment)

if completed.returncode != 0:
    raise RuntimeError(f"t_ocr.py failed with exit code {completed.returncode}")

cleanup_ocr_artifacts(CHECKPOINT_JSONL, REPO_DIR)
print(f"Download the completed file from: {OUTPUT_JSON}")
print(f"Failure log kept at: {FAILURES_JSONL}")
print("Temporary checkpoint and cloned DeepDoc repository deleted.")


/usr/local/lib/python3.12/dist-packages/gdown/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Processing sorted images 103747 through 207492 (inclusive)
Using 2 GPUs, 64 frames per hybrid group, and recognition batches of 1024 crops
Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:11<00:00, 52.0MB/s]


Handled 100 keyframes (100 resumed, 0 failed)
Handled 200 keyframes (200 resumed, 0 failed)
Handled 300 keyframes (300 resumed, 0 failed)
Handled 400 keyframes (400 resumed, 0 failed)
Handled 500 keyframes (500 resumed, 0 failed)
Handled 600 keyframes (600 resumed, 0 failed)
Handled 700 keyframes (700 resumed, 0 failed)
Handled 800 keyframes (800 resumed, 0 failed)
Handled 900 keyframes (900 resumed, 0 failed)
Handled 1000 keyframes (1000 resumed, 0 failed)
Handled 1100 keyframes (1100 resumed, 0 failed)
Handled 1200 keyframes (1200 resumed, 0 failed)
Handled 1300 keyframes (1300 resumed, 0 failed)
Handled 1400 keyframes (1400 resumed, 0 failed)
Handled 1500 keyframes (1500 resumed, 0 failed)
Handled 1600 keyframes (1600 resumed, 0 failed)
Handled 1700 keyframes (1700 resumed, 0 failed)
Handled 1800 keyframes (1800 resumed, 0 failed)
Handled 1900 keyframes (1900 resumed, 0 failed)
Handled 2000 keyframes (2000 resumed, 0 failed)
Handled 2100 keyframes (2100 resumed, 0 failed)
Handled 22

/usr/local/lib/python3.12/dist-packages/vietocr/tool/translate.py:115: RuntimeWarning: invalid value encountered in divide
  char_probs = np.sum(char_probs, axis=-1)/(char_probs>0).sum(-1)


GPU 0 | batch 40 | frames 64 | crops 608 | detect 2.51s | recognize 1.78s | total 5.49s
Handled 72256 keyframes (67072 resumed, 0 failed)
GPU 1 | batch 41 | frames 64 | crops 250 | detect 2.33s | recognize 1.09s | total 4.64s
Handled 72320 keyframes (67072 resumed, 0 failed)
GPU 0 | batch 41 | frames 64 | crops 241 | detect 2.34s | recognize 1.14s | total 4.66s
GPU 1 | batch 42 | frames 64 | crops 571 | detect 2.54s | recognize 2.30s | total 6.04s
Handled 72448 keyframes (67072 resumed, 0 failed)
GPU 0 | batch 42 | frames 64 | crops 591 | detect 2.71s | recognize 2.06s | total 6.08s
Handled 72512 keyframes (67072 resumed, 0 failed)
GPU 1 | batch 43 | frames 64 | crops 289 | detect 2.38s | recognize 1.48s | total 4.96s
GPU 0 | batch 43 | frames 64 | crops 289 | detect 2.40s | recognize 1.33s | total 4.87s
Handled 72640 keyframes (67072 resumed, 0 failed)
GPU 1 | batch 44 | frames 64 | crops 433 | detect 2.55s | recognize 2.04s | total 5.76s
Handled 72704 keyframes (67072 resumed, 0 fail